# Pipeline CAH → TextureSAM
Segmentation texture MEB par clustering hiérarchique Ward (Stage 3) puis prompting SAM2.

**Ordre d'exécution :** Setup → Modèle → Fonctions → CAH Stage 3 → Cellule 12 (K) → Cellule 13 (SAM)

In [ ]:
# ── Setup : imports, chemins, constantes ─────────────────────────────────────
import os, sys, time, warnings
import numpy as np
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import scipy.ndimage
import scipy.cluster.hierarchy as sch
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

ROOT          = Path('..').resolve()
SAM2_DIR      = ROOT / 'TextureSAM' / 'sam2'
IMG_DIR       = ROOT / 'Image_Ouassim'
CKPT_PATH     = ROOT / 'checkpoints' / 'sam2.1_hiera_small_1.pt'
MODEL_CFG_ABS = str(SAM2_DIR / 'sam2' / 'configs' / 'sam2.1' / 'sam2.1_hiera_s.yaml')

if str(SAM2_DIR) not in sys.path:
    sys.path.insert(0, str(SAM2_DIR))

SEED        = 42
IMG_SIZE    = 1024
STAGE_NAMES = ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']
CONV_IDX    = {'Stage 1': 3, 'Stage 2': 2, 'Stage 3': 1, 'Stage 4': 0}
STAGE_SIZES = {'Stage 1': 256, 'Stage 2': 128, 'Stage 3': 64, 'Stage 4': 32}

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

np.random.seed(SEED)
torch.manual_seed(SEED)

STATE  = {'features': {}, 'cah': {}}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'ROOT   : {ROOT}')
print(f'Device : {device}')
print('Setup OK')

In [ ]:
# ── Chargement SAM2 (encodeur + predictor) ────────────────────────────────────
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Le prefixe '//' indique un chemin absolu a hydra (convention TextureSAM)
_cfg_hydra = '//' + MODEL_CFG_ABS

print(f'Config : {MODEL_CFG_ABS}')
print(f'Ckpt   : {CKPT_PATH}')
print('Chargement...')

sam2_model = build_sam2(_cfg_hydra, str(CKPT_PATH),
                         device=device, apply_postprocessing=False)

# Predictor SAM2 pour le prompting (Cell 13)
predictor = SAM2ImagePredictor(sam_model=sam2_model)
STATE['predictor'] = predictor

# Encodeur image pour l'extraction de features (hooks multi-stage)
encoder = sam2_model.image_encoder
encoder.eval()
STATE['encoder'] = encoder

_hooks_pip = []

def _make_hook(sn):
    def _h(module, inp, out):
        STATE['features'][sn] = out.detach()
    return _h

for sn, ci in CONV_IDX.items():
    h = encoder.neck.convs[ci].register_forward_hook(_make_hook(sn))
    _hooks_pip.append(h)

print(f'SAM2 charge OK  (device={device})')
print(f'Hooks enregistres : {list(CONV_IDX.keys())}')
print(f'Predictor : {type(predictor).__name__}')

In [ ]:
# ── Fonctions d'extraction ────────────────────────────────────────────────────

def preprocess_image(img_path):
    img    = Image.open(img_path).convert('L')
    ow, oh = img.size
    rgb    = Image.merge('RGB', [img, img, img])
    rgb    = rgb.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    x      = torch.from_numpy(np.array(rgb)).float() / 255.0
    x      = x.permute(2, 0, 1)
    x      = (x - _MEAN) / _STD
    return x.unsqueeze(0).to(device), IMG_SIZE / ow, IMG_SIZE / oh


def extract_features(img_path):
    tensor, sx, sy = preprocess_image(img_path)
    with torch.no_grad():
        encoder(tensor)
    return {sn: STATE['features'][sn][0].permute(1, 2, 0).cpu().numpy()
            for sn in STAGE_NAMES}, sx, sy


print('Fonctions extraction OK')

In [ ]:
# ── CAH Stage 3 ───────────────────────────────────────────────────────────────

# ╔═════════════════════════════════════════╗
# ║  IMAGE A ANALYSER — MODIFIER ICI       ║
# ╚═════════════════════════════════════════╝
TARGET_IMG = '310120-pat18-WholeMount-24.tif'    # <- changer ici

# Verifier existence
assert (IMG_DIR / TARGET_IMG).exists(), f'Image introuvable : {IMG_DIR / TARGET_IMG}'

# ── Extraction features (cache) ───────────────────────────────────────────────
_ck4 = f'_pip_{TARGET_IMG}'
if _ck4 not in STATE:
    print(f'Forward pass SAM2 : {TARGET_IMG}')
    _feats4, _sx4, _sy4 = extract_features(IMG_DIR / TARGET_IMG)
    STATE[_ck4] = {'feats': _feats4, 'sx': _sx4, 'sy': _sy4}
    print('  Mis en cache.')
else:
    _feats4 = STATE[_ck4]['feats']
    print(f'Cache utilise : {TARGET_IMG}')

# ── CAH sur Stage 3 uniquement ────────────────────────────────────────────────
_MAX4   = 4096
_N_LAST = 30
_rng4   = np.random.default_rng(SEED)

_fm4        = _feats4['Stage 3']                       # (64, 64, 256)
_H4, _W4, _C4 = _fm4.shape
_V4_all     = _fm4.reshape(-1, _C4).astype(np.float32)
_N4_all     = _H4 * _W4

if _N4_all > _MAX4:
    _sidx4 = np.sort(_rng4.choice(_N4_all, size=_MAX4, replace=False))
    _V4, _N4 = _V4_all[_sidx4], _MAX4
else:
    _sidx4 = np.arange(_N4_all)
    _V4, _N4 = _V4_all.copy(), _N4_all

_nr4 = np.linalg.norm(_V4, axis=1, keepdims=True)
_V4  = _V4 / np.where(_nr4 < 1e-8, 1.0, _nr4)

_n_max4   = min(80, _N4 - 1, _C4)
_pca4     = PCA(n_components=_n_max4, random_state=SEED)
_V4_fit   = _pca4.fit_transform(_V4)
_cumvar4  = np.cumsum(_pca4.explained_variance_ratio_)
_n_comp4  = max(10, min(_n_max4, int(np.searchsorted(_cumvar4, 0.95)) + 1))
_V4_pca   = _V4_fit[:, :_n_comp4]
print(f'PCA 256 -> {_n_comp4} composantes (95% variance)')

_Z4 = sch.linkage(_V4_pca, method='ward', metric='euclidean')

# Candidats K naturels
_h4_last   = _Z4[-_N_LAST:, 2]
_gaps4_rel = np.diff(_h4_last) / np.where(_h4_last[:-1] < 1e-10, 1e-10, _h4_last[:-1])
_top4_gi   = np.argsort(_gaps4_rel)[-3:][::-1]
_cands4, _seen4 = [], set()
for _gi4 in _top4_gi:
    _kc4 = _N_LAST - int(_gi4)
    _gc4 = float(100 * _gaps4_rel[_gi4])
    if _kc4 >= 2 and _kc4 not in _seen4:
        _cands4.append((_kc4, _gc4))
        _seen4.add(_kc4)

STATE['cah']['Stage 3'] = {
    'Z': _Z4, 'n_points': _N4, 'pca_dim': _n_comp4,
    'vecteurs': _V4, 'pca_model': _pca4,
    'idx': _sidx4, 'grid_shape': (_H4, _W4), 'candidates': _cands4,
}
STATE['cah']['target_img'] = TARGET_IMG

print(f'CAH Stage 3 OK : {_N4} vecteurs -> PCA {_n_comp4}d -> linkage Ward')
print('Candidats K : ' + ', '.join(f'K={k} (gap={g:.1f}%)' for k, g in _cands4))

In [ ]:
# ── Cellule 12 — Top 3 K naturels (automatique, pas de K_CHOSEN manuel) ───────

_c12_cah3    = STATE['cah']['Stage 3']
_c12_Z       = _c12_cah3['Z']
_c12_cands   = _c12_cah3['candidates']   # [(K, gap%), ...] déjà triés gap décroissant
_c12_H, _c12_W = _c12_cah3['grid_shape']
_c12_idx     = _c12_cah3['idx']

# ── Compléter à 3 si besoin ───────────────────────────────────────────────────
_c12_top3 = list(_c12_cands[:3])
if len(_c12_top3) == 0:
    _c12_top3 = [(3, None), (2, None), (5, None)]
elif len(_c12_top3) == 1:
    _c12_k0 = _c12_top3[0][0]
    _c12_top3 += [(2, None), (_c12_k0 + 2, None)]
elif len(_c12_top3) == 2:
    _c12_k0 = _c12_top3[0][0]
    _c12_top3 += [(_c12_k0 + 2, None)]

# ── Stockage ──────────────────────────────────────────────────────────────────
STATE['k_top3']    = _c12_top3
STATE['expert_k']  = _c12_top3[0][0]
STATE['seg_target'] = TARGET_IMG          # ← nécessaire pour Cell 13

# ── Print console ─────────────────────────────────────────────────────────────
print('Top 3 K naturels (triés par gap décroissant) :')
for _c12_i, (_c12_k, _c12_g) in enumerate(_c12_top3):
    _c12_g_str = f'{_c12_g:.1f}%' if _c12_g is not None else '—'
    _c12_note  = '← gap le plus élevé' if _c12_i == 0 else ''
    print(f'  {["1er","2ème","3ème"][_c12_i]} : K={_c12_k}  gap={_c12_g_str}  {_c12_note}')
print(f'\nSTATE[\'expert_k\'] = {STATE["expert_k"]}  |  seg_target = {TARGET_IMG}')

# ── Figure 1×4 ────────────────────────────────────────────────────────────────
_c12_img  = np.array(Image.open(IMG_DIR / TARGET_IMG).convert('L'))
_c12_oh, _c12_ow = _c12_img.shape[:2]
_c12_tab  = plt.cm.get_cmap('tab10')

fig12, axes12 = plt.subplots(1, 4, figsize=(24, 7))

axes12[0].imshow(_c12_img, cmap='gray')
axes12[0].set_title('Image originale', fontsize=11)
axes12[0].axis('off')

for _c12_col, (_c12_k, _c12_g) in enumerate(_c12_top3, start=1):
    _c12_ax    = axes12[_c12_col]
    _c12_k_cut = max(2, min(_c12_k, len(_c12_Z)))
    _c12_lbl   = sch.fcluster(_c12_Z, _c12_k_cut, criterion='maxclust') - 1
    _c12_lgrid = np.full(_c12_H * _c12_W, -1, dtype=int)
    _c12_lgrid[_c12_idx] = _c12_lbl
    _c12_lgrid = _c12_lgrid.reshape(_c12_H, _c12_W)
    _c12_lup   = np.array(
        Image.fromarray((_c12_lgrid + 1).astype(np.uint8)).resize(
            (_c12_ow, _c12_oh), Image.NEAREST)
    ) - 1

    _c12_clrs = np.array([_c12_tab(c / max(_c12_k_cut, 1))[:3]
                           for c in range(_c12_k_cut)])
    _c12_rgb  = np.zeros((*_c12_lup.shape, 3), dtype=float)
    for _c12_c in range(_c12_k_cut):
        _c12_m = _c12_lup == _c12_c
        if _c12_m.any():
            _c12_rgb[_c12_m] = _c12_clrs[_c12_c]

    _c12_ax.imshow(_c12_img, cmap='gray')
    _c12_ax.imshow(_c12_rgb, alpha=0.5)
    _c12_hdls = [mpatches.Patch(color=_c12_clrs[c], label=f'C{c}')
                 for c in range(_c12_k_cut)]
    _c12_ax.legend(handles=_c12_hdls, fontsize=7, loc='upper right',
                   ncol=2 if _c12_k_cut > 5 else 1)

    _c12_g_str = f'{_c12_g:.1f}%' if _c12_g is not None else '—'
    if _c12_col == 1:
        _c12_ttl = f'K={_c12_k}  gap={_c12_g_str}\n★ Gap le plus élevé'
        for _sp in _c12_ax.spines.values():
            _sp.set_edgecolor('green'); _sp.set_linewidth(4)
    elif _c12_col == 2:
        _c12_ttl = f'K={_c12_k}  gap={_c12_g_str}\n2ème candidat'
    else:
        _c12_ttl = f'K={_c12_k}  gap={_c12_g_str}\n3ème candidat'

    _c12_ax.set_title(_c12_ttl, fontsize=10,
                      color='green' if _c12_col == 1 else 'black',
                      fontweight='bold' if _c12_col == 1 else 'normal')
    _c12_ax.axis('off')

plt.suptitle(f'Top 3 K naturels — {TARGET_IMG}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cellule 13 — Points d'inclusion + TextureSAM ─────────────────────────────
_t13 = time.time()

_K13    = STATE['expert_k']
_cah13  = STATE['cah']['Stage 3']
_Z13    = _cah13['Z']
_V13    = _cah13['vecteurs']             # (N, 256) L2-norm
_idx13  = _cah13['idx']                  # (N,)
_H13, _W13 = _cah13['grid_shape']        # 64, 64
_tgt13  = STATE['seg_target']

ALPHA13, BETA13, GAMMA13 = 0.5, 0.3, 0.2
N_INC13 = 3

# ── Etape 1 : Labels CAH au K choisi ─────────────────────────────────────────
_lflat13 = sch.fcluster(_Z13, _K13, criterion='maxclust') - 1   # (N,) 0-indexe
_lgrid13 = np.full(_H13 * _W13, -1, dtype=int)
_lgrid13[_idx13] = _lflat13
_lgrid13 = _lgrid13.reshape(_H13, _W13)                         # (64, 64)

# ── Etape 2 : Score pour chaque point de chaque cluster ──────────────────────
_inc13 = {}    # k -> (N_INC, 2) coords pixel (x, y)
_med13 = {}    # k -> coord pixel (x, y) du medoide

for _k13 in range(_K13):
    _mask13  = (_lflat13 == _k13)
    _V13_k   = _V13[_mask13]
    _idx13_k = np.where(_mask13)[0]
    _rows13  = _idx13_k // _W13
    _cols13  = _idx13_k % _W13

    if len(_V13_k) == 0:
        _inc13[_k13] = np.zeros((N_INC13, 2), dtype=int)
        _med13[_k13] = np.array([0, 0])
        continue

    # s_embed
    _ctr13   = _V13_k.mean(axis=0)
    _ctr13   = _ctr13 / (np.linalg.norm(_ctr13) + 1e-8)
    _s_emb13 = np.clip(_V13_k @ _ctr13, 0, 1)

    # s_voisinage
    _s_vois13 = np.zeros(len(_idx13_k))
    for _pi13, (_ri13, _ci13_) in enumerate(zip(_rows13, _cols13)):
        _nbrs13 = [(_ri13-1,_ci13_),(_ri13+1,_ci13_),(_ri13,_ci13_-1),(_ri13,_ci13_+1)]
        _val13  = [_lgrid13[_r,_c]==_k13 for _r,_c in _nbrs13
                   if 0<=_r<_H13 and 0<=_c<_W13]
        if _val13:
            _s_vois13[_pi13] = np.mean(_val13)

    # s_frontiere
    _dist13  = scipy.ndimage.distance_transform_edt((_lgrid13 == _k13).astype(np.uint8))
    _dv13    = _dist13[_rows13, _cols13]
    _s_frt13 = _dv13 / (_dv13.max() + 1e-8)

    _score13 = ALPHA13*_s_emb13 + BETA13*_s_vois13 + GAMMA13*_s_frt13

    _n_eff13 = min(N_INC13, len(_score13))
    _top13   = np.argsort(_score13)[-_n_eff13:]
    _inc13[_k13] = np.stack([
        (_cols13[_top13] / _W13 * 1280).astype(int),
        (_rows13[_top13] / _H13 * 768).astype(int)
    ], axis=1)

    _med_i13 = np.argmax((_V13_k @ _V13_k.T).mean(axis=1)) if len(_V13_k) > 1 else 0
    _med13[_k13] = np.array([
        int(_cols13[_med_i13] / _W13 * 1280),
        int(_rows13[_med_i13] / _H13 * 768)
    ])

_exc13 = {k: np.array([_med13[j] for j in range(_K13) if j != k])
           for k in range(_K13)}

STATE['prompts'] = {
    'inclusion': _inc13, 'exclusion': _exc13,
    'medoids': _med13, 'K': _K13, 'labels_grid': _lgrid13,
}

# ── Etape 3 : Visualisation des points avant SAM ─────────────────────────────
_img13       = np.array(Image.open(IMG_DIR / _tgt13).convert('L'))
_oh13, _ow13 = _img13.shape[:2]
_tab13       = plt.cm.get_cmap('tab10')
_clrs13      = np.array([_tab13(c / max(_K13, 1))[:3] for c in range(_K13)])

_lg_up13 = np.array(
    Image.fromarray((_lgrid13 + 1).astype(np.uint8)).resize((_ow13, _oh13), Image.NEAREST)
) - 1
_ov13 = np.zeros((*_lg_up13.shape, 3), dtype=float)
for _c13 in range(_K13):
    _m = _lg_up13 == _c13
    if _m.any():
        _ov13[_m] = _clrs13[_c13]

fig13a, (ax13L, ax13R) = plt.subplots(1, 2, figsize=(18, 7))

ax13L.imshow(_img13, cmap='gray')
ax13L.imshow(_ov13, alpha=0.4)
for _k13 in range(_K13):
    _ic = _inc13[_k13]
    ax13L.scatter(_ic[:, 0], _ic[:, 1], s=150, marker='*',
                  c=[_clrs13[_k13]], edgecolors='black', linewidths=1, zorder=5)
    _md = _med13[_k13]
    ax13L.scatter(_md[0], _md[1], s=80, marker='x', c='red', linewidths=2, zorder=5)
ax13L.set_title(f"Inclusion (*) et exclusion/medoides (x)  K={_K13}", fontsize=11)
ax13L.axis('off')

ax13R.imshow(_img13, cmap='gray')
for _k13 in range(_K13):
    _ic = _inc13[_k13]
    ax13R.scatter(_ic[:, 0], _ic[:, 1], s=200, marker='*',
                  c=[_clrs13[_k13]], edgecolors='black', linewidths=1.5, zorder=5)
    for _px13, _py13 in _ic:
        ax13R.annotate(f'C{_k13}', (_px13, _py13), xytext=(5, 5),
                       textcoords='offset points', fontsize=8,
                       color='white', fontweight='bold')
ax13R.set_title(f'Points transmis a TextureSAM (K={_K13})', fontsize=11)
ax13R.axis('off')

plt.tight_layout()
plt.show()

# ── Etape 4 : K passes TextureSAM ────────────────────────────────────────────
_pred13   = STATE['predictor']
_imgrgb13 = np.array(Image.open(IMG_DIR / _tgt13).convert('RGB'))

try:
    _pred13.set_image(_imgrgb13)
except Exception:
    pass

_masks13 = {}

for _k13 in tqdm(range(_K13), desc='TextureSAM passes'):
    _pts13  = np.vstack([_inc13[_k13], _exc13[_k13]])
    _lbls13 = np.array([1]*len(_inc13[_k13]) + [0]*len(_exc13[_k13]))
    try:
        _ms13, _ss13, _ = _pred13.predict(
            point_coords=_pts13, point_labels=_lbls13, multimask_output=False)
        _masks13[_k13] = _ms13[0].astype(bool)   # SAM retourne float → cast bool
    except Exception as _e13:
        print(f'  Cluster {_k13} : erreur SAM -> {_e13}')
        _masks13[_k13] = np.zeros((_oh13, _ow13), dtype=bool)

STATE['masks'] = _masks13

# ── Etape 5 : Affichage resultat final ───────────────────────────────────────
_ncols13 = max(_K13, 2)
fig13b, axes13b = plt.subplots(2, _ncols13, figsize=(max(16, _K13*4), 10))
if _ncols13 == 1:
    axes13b = axes13b[:, None]

# Ligne 1 : masque binaire de chaque cluster
for _k13 in range(_K13):
    _ax1 = axes13b[0, _k13]
    _m13 = _masks13.get(_k13, np.zeros((_oh13, _ow13), dtype=bool))
    _m13 = np.asarray(_m13, dtype=bool)          # securite : garantit bool
    _ax1.imshow(_img13, cmap='gray')
    _mrgb13 = np.zeros((*_m13.shape, 4), dtype=float)
    _mrgb13[_m13] = [1, 1, 1, 0.7]
    _ax1.imshow(_mrgb13)
    _ax1.set_title(f'Cluster {_k13} — masque SAM', fontsize=9)
    _ax1.axis('off')
for _x13 in range(_K13, _ncols13):
    axes13b[0, _x13].axis('off')
    axes13b[1, _x13].axis('off')

# Ligne 2 : original + superposition finale
axes13b[1, 0].imshow(_img13, cmap='gray')
axes13b[1, 0].set_title('Image originale', fontsize=10)
axes13b[1, 0].axis('off')

if _ncols13 > 1:
    _axf13 = axes13b[1, 1]
    _axf13.imshow(_img13, cmap='gray')
    for _k13 in range(_K13):
        _m13 = np.asarray(_masks13.get(_k13, np.zeros((_oh13, _ow13))), dtype=bool)
        if not _m13.any():
            continue
        _ov_rgba = np.zeros((*_m13.shape, 4), dtype=float)
        _ov_rgba[_m13] = [*_clrs13[_k13], 0.6]
        _axf13.imshow(_ov_rgba)
        _axf13.contour(_m13.astype(float), levels=[0.5],
                       colors=[_clrs13[_k13]], linewidths=1.5)
    _axf13.set_title(f'Segmentation finale TextureSAM — K={_K13} clusters', fontsize=10)
    _axf13.axis('off')

plt.suptitle(f'Resultat pipeline complet — {_tgt13}  K={_K13}', fontsize=12)
plt.tight_layout()
plt.show()

# ── Print console final ──────────────────────────────────────────────────────
_elapsed13 = time.time() - _t13
print()
for _k13 in range(_K13):
    _m13 = np.asarray(_masks13.get(_k13, np.zeros((_oh13, _ow13))), dtype=bool)
    _aire13 = int(_m13.sum())
    _pct13  = 100 * _aire13 / (_oh13 * _ow13)
    print(f'Cluster {_k13} : aire = {_aire13:,} px ({_pct13:.1f}% image)')
print(f'Pipeline complet en {_elapsed13:.1f}s')

In [ ]:
# ── Cellule 14 — Visualisation des résultats (sans recalcul) ─────────────────

_K14      = STATE['expert_k']
_masks14  = STATE['masks']
_tgt14    = STATE['seg_target']

_img14_gray = np.array(Image.open(IMG_DIR / _tgt14).convert('L'))
_img14_rgb  = np.array(Image.open(IMG_DIR / _tgt14).convert('RGB'))
_oh14, _ow14 = _img14_gray.shape[:2]

_tab14  = plt.cm.get_cmap('tab10')
_clrs14 = np.array([_tab14(k / max(_K14, 1))[:3] for k in range(_K14)])

# Pré-calcul des métriques par cluster
_aires14 = {}
for _k14 in range(_K14):
    _m14 = np.asarray(_masks14.get(_k14, np.zeros((_oh14, _ow14))), dtype=bool)
    _aires14[_k14] = (_m14.sum(), 100 * _m14.sum() / (_oh14 * _ow14))

# ── Figure principale : K lignes × 3 colonnes ────────────────────────────────
fig14a, axes14a = plt.subplots(_K14, 3, figsize=(18, 4.5 * _K14))
if _K14 == 1:
    axes14a = axes14a[None, :]   # garantir tableau 2D

for _k14 in range(_K14):
    _m14      = np.asarray(_masks14.get(_k14, np.zeros((_oh14, _ow14))), dtype=bool)
    _aire_px, _aire_pct = _aires14[_k14]
    _col14    = _clrs14[_k14]

    # ── Col 0 : cluster isolé ────────────────────────────────────────────────
    _rgba14_iso = np.zeros((_oh14, _ow14, 4), dtype=float)
    _rgba14_iso[_m14] = [*_col14, 0.6]

    axes14a[_k14, 0].imshow(_img14_gray, cmap='gray')
    axes14a[_k14, 0].imshow(_rgba14_iso)
    axes14a[_k14, 0].set_title(f"Cluster {_k14}  —  {_aire_pct:.1f}% de l'image",
                                fontsize=10)
    axes14a[_k14, 0].axis('off')

    # ── Col 1 : texture extraite (fond très sombre) ──────────────────────────
    _tex14 = _img14_gray.astype(float)
    _tex14_out = (_tex14 * 0.15).astype(np.uint8)
    _tex14_in  = _tex14.astype(np.uint8)
    _tex14_disp = np.where(_m14, _tex14_in, _tex14_out)

    axes14a[_k14, 1].imshow(_tex14_disp, cmap='gray', vmin=0, vmax=255)
    axes14a[_k14, 1].set_title('Texture extraite', fontsize=10)
    axes14a[_k14, 1].axis('off')

    # ── Col 2 : crop serré ───────────────────────────────────────────────────
    if _m14.any():
        _rows14 = np.where(np.any(_m14, axis=1))[0]
        _cols14 = np.where(np.any(_m14, axis=0))[0]
        _r0, _r1 = _rows14[0], _rows14[-1]
        _c0, _c1 = _cols14[0], _cols14[-1]
    else:
        _r0, _r1, _c0, _c1 = 0, _oh14 - 1, 0, _ow14 - 1

    _crop14_gray = _img14_gray[_r0:_r1+1, _c0:_c1+1]
    _crop14_mask = _m14[_r0:_r1+1, _c0:_c1+1]
    _crop14_ov   = np.zeros((*_crop14_gray.shape, 4), dtype=float)
    _crop14_ov[_crop14_mask] = [*_col14, 0.5]

    axes14a[_k14, 2].imshow(_crop14_gray, cmap='gray')
    axes14a[_k14, 2].imshow(_crop14_ov)
    axes14a[_k14, 2].set_title(f'Crop  {_c1-_c0}×{_r1-_r0} px', fontsize=10)
    axes14a[_k14, 2].axis('off')

fig14a.suptitle(f'Détail par cluster — {_tgt14}', fontsize=12)
fig14a.subplots_adjust(hspace=0.35)

# ── Figure résumé : 1×2 ──────────────────────────────────────────────────────
fig14b, (ax14_orig, ax14_comp) = plt.subplots(1, 2, figsize=(16, 7))

# Col 0 : image originale
ax14_orig.imshow(_img14_gray, cmap='gray')
ax14_orig.set_title('Image originale', fontsize=11)
ax14_orig.axis('off')

# Col 1 : segmentation composite
ax14_comp.imshow(_img14_gray, cmap='gray')

# Peindre chaque cluster + contour (ordre : plus petites aires d'abord)
_order14 = sorted(range(_K14), key=lambda k: _aires14[k][0])
for _k14 in _order14:
    _m14 = np.asarray(_masks14.get(_k14, np.zeros((_oh14, _ow14))), dtype=bool)
    if not _m14.any():
        continue
    _ov14 = np.zeros((*_m14.shape, 4), dtype=float)
    _ov14[_m14] = [*_clrs14[_k14], 0.55]
    ax14_comp.imshow(_ov14)
    ax14_comp.contour(_m14.astype(float), levels=[0.5],
                      colors=[_clrs14[_k14]], linewidths=1.5)

# Légende triée par aire décroissante
_handles14 = [
    mpatches.Patch(
        color=_clrs14[_k14],
        label=f'Cluster {_k14} — {_aires14[_k14][1]:.1f}%'
    )
    for _k14 in sorted(range(_K14), key=lambda k: -_aires14[k][0])
]
ax14_comp.legend(handles=_handles14, fontsize=9, loc='upper right',
                 framealpha=0.8)
ax14_comp.set_title(f'Segmentation TextureSAM — K={_K14} textures', fontsize=11)
ax14_comp.axis('off')

fig14b.suptitle(_tgt14, fontsize=11)
fig14b.tight_layout()

plt.show()

In [ ]:
# ── Cellule 14 — Random prompting + TextureSAM ───────────────────────────────
import scipy.cluster.hierarchy as sch

if 'prompts' not in STATE:
    print('Lance Cell 13 d\'abord (exclusion points requis).')
else:
    _c14_K      = STATE['expert_k']
    _c14_cah3   = STATE['cah']['Stage 3']
    _c14_Z3     = _c14_cah3['Z']
    _c14_H3, _c14_W3 = _c14_cah3['grid_shape']   # 64, 64
    _c14_idx3   = _c14_cah3['idx']
    _c14_tgt    = STATE.get('seg_target', TARGET_IMG)
    _c14_N_INC  = 3

    # Labels CAH au K choisi
    _c14_labels = sch.fcluster(_c14_Z3, _c14_K, criterion='maxclust') - 1
    _c14_lgrid  = np.full(_c14_H3 * _c14_W3, -1, dtype=int)
    _c14_lgrid[_c14_idx3] = _c14_labels
    _c14_lgrid  = _c14_lgrid.reshape(_c14_H3, _c14_W3)

    # Dimensions image depuis le fichier
    _c14_img_gray = np.array(Image.open(IMG_DIR / _c14_tgt).convert('L'))
    _c14_oh, _c14_ow = _c14_img_gray.shape[:2]

    # ── Tirage aléatoire N_INC points par cluster ─────────────────────────────
    _c14_rng = np.random.default_rng(SEED)
    _c14_inc_random = {}

    for _c14_k in range(_c14_K):
        _c14_flat = (_c14_lgrid.flatten() == _c14_k)
        _c14_rows_k = np.where(_c14_flat)[0] // _c14_W3
        _c14_cols_k = np.where(_c14_flat)[0] % _c14_W3
        _c14_nk = len(_c14_rows_k)
        if _c14_nk == 0:
            _c14_inc_random[_c14_k] = np.zeros((_c14_N_INC, 2), dtype=int)
            continue
        _c14_sel = _c14_rng.choice(_c14_nk, size=_c14_N_INC,
                                    replace=(_c14_nk < _c14_N_INC))
        _c14_inc_random[_c14_k] = np.stack([
            (_c14_cols_k[_c14_sel] / _c14_W3 * _c14_ow).astype(int),
            (_c14_rows_k[_c14_sel] / _c14_H3 * _c14_oh).astype(int),
        ], axis=1)

    # ── K passes TextureSAM ───────────────────────────────────────────────────
    _c14_exc_pts = STATE['prompts']['exclusion']
    _c14_img_rgb = np.array(Image.open(IMG_DIR / _c14_tgt).convert('RGB'))
    try:
        predictor.set_image(_c14_img_rgb)
    except Exception:
        pass

    _c14_masks_random = {}

    for _c14_k in tqdm(range(_c14_K), desc='Random SAM passes'):
        _c14_inc = _c14_inc_random[_c14_k]
        _c14_exc = np.asarray(_c14_exc_pts.get(_c14_k, np.zeros((0, 2), dtype=int)))
        if len(_c14_exc) > 0:
            _c14_pts  = np.vstack([_c14_inc, _c14_exc])
            _c14_lbls = np.array([1]*_c14_N_INC + [0]*len(_c14_exc))
        else:
            _c14_pts  = _c14_inc
            _c14_lbls = np.array([1]*_c14_N_INC)
        try:
            _c14_ms, _, _ = predictor.predict(
                point_coords=_c14_pts, point_labels=_c14_lbls,
                multimask_output=False)
            _c14_masks_random[_c14_k] = _c14_ms[0].astype(bool)
        except Exception as _e14:
            print(f'  Cluster {_c14_k} : {_e14}')
            _c14_masks_random[_c14_k] = np.zeros((_c14_oh, _c14_ow), dtype=bool)

    STATE['masks_random'] = _c14_masks_random
    STATE['inc_random']   = _c14_inc_random

    # ── Figure 1×(K+1) ────────────────────────────────────────────────────────
    _c14_tab  = plt.cm.get_cmap('tab10')
    _c14_clrs = np.array([_c14_tab(c / max(_c14_K, 1))[:3] for c in range(_c14_K)])

    fig14, axes14 = plt.subplots(1, _c14_K + 1, figsize=(4 * (_c14_K + 1), 5))
    axes14 = np.atleast_1d(axes14).flatten()

    axes14[0].imshow(_c14_img_gray, cmap='gray')
    axes14[0].set_title('Image originale', fontsize=10)
    axes14[0].axis('off')

    for _c14_k in range(_c14_K):
        _c14_ax = axes14[_c14_k + 1]
        _c14_m  = np.asarray(_c14_masks_random.get(_c14_k,
                              np.zeros((_c14_oh, _c14_ow))), dtype=bool)
        _c14_ov = np.zeros((*_c14_m.shape, 4), dtype=float)
        _c14_ov[_c14_m] = [*_c14_clrs[_c14_k], 0.6]
        _c14_ax.imshow(_c14_img_gray, cmap='gray')
        _c14_ax.imshow(_c14_ov)
        _c14_inc = _c14_inc_random[_c14_k]
        _c14_ax.scatter(_c14_inc[:, 0], _c14_inc[:, 1],
                        s=120, marker='*', c='white',
                        edgecolors='black', linewidths=1, zorder=5)
        _c14_ax.set_title(f'Cluster {_c14_k} — Random', fontsize=9)
        _c14_ax.axis('off')

    plt.suptitle(f'Random prompting — K={_c14_K}', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Random : {_c14_K} masques générés')

In [ ]:
# ── Cellule 15 — Max Distance prompting + TextureSAM ─────────────────────────
# Farthest Point Sampling : P1=centroïde, P2=plus loin de P1,
#                           P3=maximise min(dist(P,P1), dist(P,P2))

if 'prompts' not in STATE:
    print('Lance Cell 13 d\'abord (exclusion points requis).')
else:
    _c15_K      = STATE['expert_k']
    _c15_cah3   = STATE['cah']['Stage 3']
    _c15_Z3     = _c15_cah3['Z']
    _c15_H3, _c15_W3 = _c15_cah3['grid_shape']
    _c15_idx3   = _c15_cah3['idx']
    _c15_tgt    = STATE.get('seg_target', TARGET_IMG)
    _c15_N_INC  = 3

    _c15_labels = sch.fcluster(_c15_Z3, _c15_K, criterion='maxclust') - 1
    _c15_lgrid  = np.full(_c15_H3 * _c15_W3, -1, dtype=int)
    _c15_lgrid[_c15_idx3] = _c15_labels
    _c15_lgrid  = _c15_lgrid.reshape(_c15_H3, _c15_W3)

    _c15_img_gray = np.array(Image.open(IMG_DIR / _c15_tgt).convert('L'))
    _c15_oh, _c15_ow = _c15_img_gray.shape[:2]

    # ── Farthest Point Sampling par cluster ───────────────────────────────────
    _c15_inc_maxdist = {}

    for _c15_k in range(_c15_K):
        _c15_flat   = (_c15_lgrid.flatten() == _c15_k)
        _c15_rows_k = np.where(_c15_flat)[0] // _c15_W3
        _c15_cols_k = np.where(_c15_flat)[0] % _c15_W3
        _c15_nk     = len(_c15_rows_k)

        if _c15_nk == 0:
            _c15_inc_maxdist[_c15_k] = np.zeros((_c15_N_INC, 2), dtype=int)
            continue

        _c15_pts = np.stack([_c15_cols_k, _c15_rows_k], axis=1).astype(float)   # (nk, 2)

        if _c15_nk == 1:
            _c15_sel_grid = np.repeat(_c15_pts, _c15_N_INC, axis=0)
        else:
            # P1 : centroïde spatial du cluster
            _c15_p1 = _c15_pts.mean(axis=0)

            # P2 : plus éloigné de P1
            _c15_d1 = np.linalg.norm(_c15_pts - _c15_p1, axis=1)
            _c15_p2 = _c15_pts[np.argmax(_c15_d1)]

            # P3 : maximise min(dist(p, p1), dist(p, p2))
            _c15_d2   = np.linalg.norm(_c15_pts - _c15_p2, axis=1)
            _c15_mind = np.minimum(_c15_d1, _c15_d2)
            _c15_p3   = _c15_pts[np.argmax(_c15_mind)]

            _c15_sel_grid = np.array([_c15_p1, _c15_p2, _c15_p3])   # (3, 2) en 64×64

        # Convertir vers pixels originaux (x = col, y = row)
        _c15_inc_maxdist[_c15_k] = np.stack([
            (_c15_sel_grid[:, 0] / _c15_W3 * _c15_ow).astype(int),
            (_c15_sel_grid[:, 1] / _c15_H3 * _c15_oh).astype(int),
        ], axis=1)

    # ── K passes TextureSAM ───────────────────────────────────────────────────
    _c15_exc_pts = STATE['prompts']['exclusion']
    _c15_img_rgb = np.array(Image.open(IMG_DIR / _c15_tgt).convert('RGB'))
    try:
        predictor.set_image(_c15_img_rgb)
    except Exception:
        pass

    _c15_masks_maxdist = {}

    for _c15_k in tqdm(range(_c15_K), desc='Max Distance SAM passes'):
        _c15_inc = _c15_inc_maxdist[_c15_k]
        _c15_exc = np.asarray(_c15_exc_pts.get(_c15_k, np.zeros((0, 2), dtype=int)))
        if len(_c15_exc) > 0:
            _c15_pts2  = np.vstack([_c15_inc, _c15_exc])
            _c15_lbls  = np.array([1]*_c15_N_INC + [0]*len(_c15_exc))
        else:
            _c15_pts2  = _c15_inc
            _c15_lbls  = np.array([1]*_c15_N_INC)
        try:
            _c15_ms, _, _ = predictor.predict(
                point_coords=_c15_pts2, point_labels=_c15_lbls,
                multimask_output=False)
            _c15_masks_maxdist[_c15_k] = _c15_ms[0].astype(bool)
        except Exception as _e15:
            print(f'  Cluster {_c15_k} : {_e15}')
            _c15_masks_maxdist[_c15_k] = np.zeros((_c15_oh, _c15_ow), dtype=bool)

    STATE['masks_maxdist'] = _c15_masks_maxdist
    STATE['inc_maxdist']   = _c15_inc_maxdist

    # ── Figure 1×(K+1) avec points colorés ───────────────────────────────────
    _c15_tab    = plt.cm.get_cmap('tab10')
    _c15_clrs   = np.array([_c15_tab(c / max(_c15_K, 1))[:3] for c in range(_c15_K)])
    _c15_pcols  = ['limegreen', 'red', 'dodgerblue']   # P1 centroïde, P2 farthest, P3 max-min
    _c15_plbls  = ['P1 centroïde', 'P2 plus loin de P1', 'P3 max-min dist']

    fig15, axes15 = plt.subplots(1, _c15_K + 1, figsize=(4 * (_c15_K + 1), 5))
    axes15 = np.atleast_1d(axes15).flatten()

    axes15[0].imshow(_c15_img_gray, cmap='gray')
    axes15[0].set_title('Image originale', fontsize=10)
    axes15[0].axis('off')
    # Légende couleurs dans la première case
    from matplotlib.lines import Line2D
    _c15_leg = [Line2D([0], [0], marker='*', color='w', markerfacecolor=_c,
                       markeredgecolor='black', markersize=10, label=_l)
                for _c, _l in zip(_c15_pcols, _c15_plbls)]
    axes15[0].legend(handles=_c15_leg, fontsize=7, loc='lower left', framealpha=0.8)

    for _c15_k in range(_c15_K):
        _c15_ax = axes15[_c15_k + 1]
        _c15_m  = np.asarray(_c15_masks_maxdist.get(_c15_k,
                              np.zeros((_c15_oh, _c15_ow))), dtype=bool)
        _c15_ov = np.zeros((*_c15_m.shape, 4), dtype=float)
        _c15_ov[_c15_m] = [*_c15_clrs[_c15_k], 0.6]
        _c15_ax.imshow(_c15_img_gray, cmap='gray')
        _c15_ax.imshow(_c15_ov)

        # Points colorés : vert=centroïde, rouge=farthest, bleu=max-min
        _c15_inc = _c15_inc_maxdist[_c15_k]
        for _c15_pi in range(min(_c15_N_INC, len(_c15_inc))):
            _c15_ax.scatter(_c15_inc[_c15_pi, 0], _c15_inc[_c15_pi, 1],
                            s=140, marker='*', color=_c15_pcols[_c15_pi],
                            edgecolors='black', linewidths=1, zorder=5)

        _c15_ax.set_title(f'Cluster {_c15_k} — Max Distance', fontsize=9)
        _c15_ax.axis('off')

    plt.suptitle(f'Max Distance prompting — K={_c15_K}', fontsize=12)
    plt.tight_layout()
    plt.show()
    print(f'Max Distance : {_c15_K} masques générés')